In [ ]:
%%sql -r dataframe_1
USE ROLE SYSADMIN;
ALTER SESSION SET query_tag = '{"Phase":"Setup","Project":"Panel Demo","version":{"major":1, "minor":1},"Organization":"TastyBytes", "source":"setup-sfpd.sql", "Section": "1","Goal":"Setting up Iceberg Tables,Connection to an external Volume"}';
session

In [ ]:
%%sql -r dataframe_2

--====Iceberg requires an external VOLUME
-- For this demo I am using my sandbox AWS bucket with required IAM permissions
-- Lets start with External VOLUME

USE role accountadmin;

CREATE OR REPLACE EXTERNAL VOLUME iceberg_external_volume
   STORAGE_LOCATIONS =
      (
         (
            NAME = 'tb-de-demo-bucket'
            STORAGE_PROVIDER = 'S3'
            STORAGE_BASE_URL = 's3://tb-de-demo-bucket'
            STORAGE_AWS_ROLE_ARN = 'arn:aws:iam::981304421142:role/tb-de-role'
            STORAGE_AWS_EXTERNAL_ID = 'iceberg_table_external_id'
         )
      )
      ALLOW_WRITES = TRUE;

//DESCRIBE external volume iceberg_external_volume;
--DESC EXTERNAL VOLUME iceberg_external_volume;

/* OUTPUT 
{"NAME":"tb-de-demo-bucket","STORAGE_PROVIDER":"S3","STORAGE_BASE_URL":"s3://tb-de-demo-bucket","STORAGE_ALLOWED_LOCATIONS":["s3://tb-de-demo-bucket/*"],"STORAGE_AWS_ROLE_ARN":"arn:aws:iam::981304421142:role/tb-de-role","STORAGE_AWS_IAM_USER_ARN":"arn:aws:iam::518289065437:user/ecsm1000-s","STORAGE_AWS_EXTERNAL_ID":"iceberg_table_external_id","ENCRYPTION_TYPE":"NONE","ENCRYPTION_KMS_KEY_ID":""} */

//	Replaced		    "AWS": "arn:aws:iam::981304421142:root"

DESC EXTERNAL VOLUME iceberg_external_volume;
SELECT 
    PARSE_JSON("property_value"):STORAGE_AWS_IAM_USER_ARN AS STORAGE_AWS_IAM_USER_ARN,
    PARSE_JSON("property_value"):STORAGE_AWS_EXTERNAL_ID AS STORAGE_AWS_EXTERNAL_ID,
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID())) WHERE "property" = 'STORAGE_LOCATION_1';

SELECT SYSTEM$VERIFY_EXTERNAL_VOLUME('iceberg_external_volume');

/* OUTPUT 
{"success":true,"storageLocationSelectionResult":"PASSED","storageLocationName":"tb-de-demo-bucket","servicePrincipalProperties":"STORAGE_AWS_IAM_USER_ARN: arn:aws:iam::518289065437:user/ecsm1000-s; STORAGE_AWS_EXTERNAL_ID: iceberg_table_external_id","location":"s3://tb-de-demo-bucket","storageAccount":null,"region":"us-east-2","writeResult":"PASSED","readResult":"PASSED","listResult":"PASSED","deleteResult":"PASSED","awsRoleArnValidationResult":"PASSED","azureGetUserDelegationKeyResult":"SKIPPED"}
*/

In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE DATABASE ice_pipe_demo_db;

CREATE OR REPLACE WAREHOUSE ice_pipe_demo_wh
    WAREHOUSE_SIZE = 'large' -- Large for initial data load - scaled down to XSmall at end of this scripts
    WAREHOUSE_TYPE = 'standard'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE
COMMENT = 'Tasty Bytes - Data Engineering Warehouse';

CREATE OR REPLACE SCHEMA ice_pipe_demo_db.raw_data_schema;
USE DATABASE ice_pipe_demo_db;
USE WAREHOUSE ice_pipe_demo_wh;
use schema ice_pipe_demo_db.raw_data_schema;

In [ ]:
%%sql -r dataframe_4
CREATE ROLE sandbox_role;
GRANT ALL ON DATABASE ice_pipe_demo_db TO ROLE sandbox_role WITH GRANT OPTION;
GRANT ALL ON SCHEMA ice_pipe_demo_db.raw_data_schema TO ROLE sandbox_role WITH GRANT OPTION;
GRANT ALL ON WAREHOUSE ice_pipe_demo_wh TO ROLE sandbox_role WITH GRANT OPTION;
GRANT ALL ON FUTURE TABLES IN SCHEMA ice_pipe_demo_db.raw_data_schema TO ROLE sandbox_role;
GRANT ALL ON FUTURE VIEWS IN SCHEMA ice_pipe_demo_db.raw_data_schema TO ROLE sandbox_role;

SET my_user = CURRENT_USER();
GRANT ROLE sandbox_role TO USER IDENTIFIER($my_user);

USE ROLE sandbox_role;

In [ ]:
%%sql -r dataframe_5
Use role accountadmin;
-- Step 2: Register the public key with your Snowflake user:
-- ALTER USER XAVIZONE SET RSA_PUBLIC_KEY='<paste contents of rsa_key.pub, no header/footer>';
ALTER USER XAVIZONE SET RSA_PUBLIC_KEY='MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEA3cRwQtwoHb360KKujIUrPKlosupED/eDxJ4xhuY9UtLOu66l378MvTHSIxSMNDgkRXJbLikYLO8ldF8aPpm3OfYmHx6n95YpYDeSwYgzFoaTEEXOczCuW4bIfWA/t6dRkzxccqVvz373KHtJwHHTwKz1dvTfjDj/uGqbfvmLaGkXrbNkscxufFtiBSYyAv01T+4OjU1NkYFdbrbVylPOb9FouNbKG69/FjFGLFuhk/giGMaM3OfhiQHJM3w1/EdHbPglMD8Yjbcoj8w1WqPvjFub32dP/qmT85hxrj2dzTQ2K7dR9gqMgwwMNmt9QSxKuctfoat0xqbHjmQ+0LJHuQIDAQAB';